## Approved faster Colab execution (31 October 2026)
Use CPU for acquisition, audit, preprocessing, spectra, extraction, evaluation and reporting. These tasks use the existing CPU libraries. Do not rerun completed phases to change runtime. Phase 5 optionally accepts `--workers 2`; outputs remain ordered, checksummed and sealed. GPU acceleration is optional for XGBoost only.

Phase 6 fast amendment: 3 shared stratified folds, 10 sampled candidates per model (180 search fits), 3-fold sigmoid SVM calibration, the same six pipelines/search ranges/seed/feature-choice grid. This is a user-approved reduction of the original 5-fold/30-candidate protocol, not equivalent evidence of optimization quality. Original configs and records remain unchanged. CPU and T4 profiles have independent outputs/checkpoints. Keep this choice fixed before evaluation. No Phase 7 authorization is inferred.

Run setup once, then these cells only. Never use Run all. Existing completed runs refuse rerunning. A stale lock requires confirmation that its old runtime stopped before removing only that lock. See `docs/COLAB_PERFORMANCE.md`.


In [ ]:
from pathlib import Path
import subprocess, sys, os, json
PROJECT_DIR = Path('/content/ai-image-fft-detector')
os.chdir(PROJECT_DIR)
FAST_CONFIG = PROJECT_DIR / 'config_r2_fast.yaml'  # CPU default; optional config_r2_fast_t4.yaml
fast_env = os.environ.copy()
fast_env['PYTHONPATH'] = str(PROJECT_DIR / 'src') + os.pathsep + str(PROJECT_DIR / 'scripts')
fast_env.update(OMP_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1', MKL_NUM_THREADS='1', NUMEXPR_NUM_THREADS='1')
# This verifies the reviewed Phase 5 record and frozen provenance; no extraction or test reads.
processes = subprocess.check_output(['ps', '-eo', 'pid,args'], text=True)
assert not any('scripts/train_models.py' in line or 'scripts/run_fast_training.py' in line for line in processes.splitlines()), 'A training process is already running'
subprocess.run([sys.executable, 'scripts/prepare_fast_training.py', '--config', str(FAST_CONFIG)], cwd=PROJECT_DIR, env=fast_env, check=True)


In [ ]:
# Optional timing only. Reads TRAIN batches; emits no research metrics.
BENCHMARK_PATH = Path('/content/drive/MyDrive/aidetect/r2/review_backup') / ('cpu_benchmark_' + __import__('datetime').datetime.now(__import__('datetime').timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '.json')
subprocess.run([sys.executable, '-u', 'scripts/benchmark_training.py', '--config', str(FAST_CONFIG), '--output', str(BENCHMARK_PATH)], cwd=PROJECT_DIR, env=fast_env, check=True)
print(BENCHMARK_PATH.read_text())


In [ ]:
# Launches Phase 6 only. The runner independently backs up all six completed models/results.
assert not ('fast_process' in globals() and fast_process.poll() is None), 'Fast training is already running'
assert not any('scripts/train_models.py' in line or 'scripts/run_fast_training.py' in line for line in subprocess.check_output(['ps', '-eo', 'pid,args'], text=True).splitlines()), 'Another training process is active'
FAST_LAUNCH_LOG = Path('/content/drive/MyDrive/aidetect/r2') / ('fast_launch_' + __import__('datetime').datetime.now(__import__('datetime').timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '.log')
with FAST_LAUNCH_LOG.open('x') as stream:
    fast_process = subprocess.Popen([sys.executable, '-u', 'scripts/run_fast_training.py', '--config', str(FAST_CONFIG)], cwd=PROJECT_DIR, env=fast_env, stdout=stream, stderr=subprocess.STDOUT, start_new_session=True)
print('Fast Phase 6 runner PID:', fast_process.pid)
print('Launch log:', FAST_LAUNCH_LOG)
print('STOP: no freeze/test evaluation queued. Use the status cell; do not rerun launch.')


In [ ]:
# Read-only status; safe while the training subprocess is running.
from pathlib import Path
import json
root = Path('/content/drive/MyDrive/aidetect/r2')
for folder in sorted((root / 'review_backup').glob('r2_fast*_training_*'))[-2:]:
    print(folder)
    for name in ('start.json', 'training.log'):
        path = folder / name
        if path.is_file(): print(name, path.read_text()[-3000:])
for folder in sorted(root.glob('r2_fast*_phase6_checkpoints')):
    path = folder / 'status.json'
    if path.is_file(): print(folder, path.read_text())
print('STOP before Phase 7. Review actual completed outputs and practical target first.')
